# 03 Retrain on train + validation: Single-stage Ridge

Part of `03b_training_pipeline_model/single_stage_ridge`; reads `feature_config.py` (notebook 01) and `model_config.py` (notebook 02).

For each training window the tuned setup is **retrained from scratch on train + validation** (2021-01 to 2025-03, restricted to the window's months): the
temporal features come from the `train_val` selection in `feature_config.py`, and the trend decomposition, the linear model and (two-stage) the random
forest are refit with the tuned settings.

The fitted models are saved to `outputs/<window>.joblib` (one `pipeline_utils.fit_model` result each, loadable with `joblib.load` and usable with
`pipeline_utils.predict_model`), so the benchmark and inference steps can run without retraining.

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

SHARED_DIR = '/Users/keltonn/Desktop/Projects/housing_price_regressor/03b_training_pipeline_model'
FOLDER = f'{SHARED_DIR}/single_stage_ridge'
sys.path.insert(0, SHARED_DIR)      # pipeline_utils.py, shared by every model folder
sys.path.insert(0, FOLDER)          # this folder's feature_config.py and model_config.py
import os
import joblib
import feature_config
import model_config
import pipeline_utils as pu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

OUT_DIR = f'{FOLDER}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Data

In [2]:
frames, numeric, onehot = pu.load_static()
temporal = pu.load_temporal()
tv = pd.concat([frames['train'], frames['validation']], ignore_index=True)
X_tv, y_tv, m_tv = pu.design(tv, numeric, onehot), pu.log_rent(tv), tv['month'].to_numpy()
print(f'train + validation {X_tv.shape}')
print(len(model_config.BEST_SETUPS), 'setups to retrain:', list(model_config.BEST_SETUPS))

train + validation (149359, 237)
3 setups to retrain: ['full_history/single_stage_ridge', 'from_2023/single_stage_ridge', 'from_2023_06/single_stage_ridge']


## 2. Retrain every window on train + validation and save

In [3]:
fitted = {}
for key, setup in model_config.BEST_SETUPS.items():
    window = key.split('/')[0]
    selected = feature_config.SELECTED_TEMPORAL_FEATURES[f'{window}/train_val']
    fitted[key] = pu.fit_model(key, setup, X_tv, y_tv, m_tv, temporal, numeric, onehot, selected)
    coef = fitted[key]['linear'].coef_
    penalty = setup.get('lasso_alpha', setup.get('ridge_alpha'))
    print(f'{key}: retrained on {int((m_tv >= pu.WINDOW_START[window]).sum()):,} rows; penalty {penalty:g}; '
          f'{int((coef != 0).sum())} of {len(coef)} linear coefficients nonzero')
    joblib.dump(fitted[key], f'{OUT_DIR}/{window}.joblib', compress=3)

full_history/single_stage_ridge: retrained on 149,359 rows; penalty 100; 596 of 597 linear coefficients nonzero


from_2023/single_stage_ridge: retrained on 79,177 rows; penalty 10000; 636 of 637 linear coefficients nonzero


from_2023_06/single_stage_ridge: retrained on 64,458 rows; penalty 5000; 636 of 637 linear coefficients nonzero


## 3. Check the saved models

Each saved file is loaded back and must reproduce the in-memory model's predictions exactly (checked on the validation rows, which are part of the
retraining data).

In [4]:
X_check, m_check = pu.design(frames['validation'], numeric, onehot), frames['validation']['month'].to_numpy()
rows = []
for key in fitted:
    window = key.split('/')[0]
    path = f'{OUT_DIR}/{window}.joblib'
    reloaded = joblib.load(path)
    same = np.allclose(pu.predict_model(fitted[key], X_check, m_check), pu.predict_model(reloaded, X_check, m_check), rtol=0, atol=1e-9)
    assert same, f'{key}: the reloaded model predicts differently'
    rows.append({'model': key, 'file': os.path.basename(path), 'size MB': round(os.path.getsize(path) / 1e6, 1), 'reload reproduces predictions': same,
                 'temporal features': len(fitted[key]['selected'])})
pd.DataFrame(rows).set_index('model')

,file,size MB,reload reproduces predictions,temporal features
model,,,,
full_history/single_stage_ridge,full_history.joblib,0.0,True,9
from_2023/single_stage_ridge,from_2023.joblib,0.0,True,10
from_2023_06/single_stage_ridge,from_2023_06.joblib,0.0,True,10


In [5]:
# in-sample reference on train + validation (every row of the window was used to fit, so this is not out-of-sample accuracy)
pd.DataFrame({key: pu.score(y_tv[m_tv >= pu.WINDOW_START[key.split('/')[0]]],
                            pu.predict_model(f, X_tv[m_tv >= pu.WINDOW_START[key.split('/')[0]]], m_tv[m_tv >= pu.WINDOW_START[key.split('/')[0]]]))
              for key, f in fitted.items()}).T.round(4)

,log_r2,dollar_r2,log_rmse,dollar_rmse
full_history/single_stage_ridge,0.5407,0.5710,0.1941,491.2301
from_2023/single_stage_ridge,0.3258,0.3804,0.2100,556.4224
from_2023_06/single_stage_ridge,0.3408,0.4042,0.2021,533.3830
